# JSON - Rust

All 9 Rust examples from [docs/json.md](https://platob.github.io/yggdryl/json/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::{Value, json};

let value = Value::from_mapping([
    (Value::from("symbol"), Value::from("AAPL")),
    (Value::from("quantity"), Value::from(100_i64)),
])?;

let encoded = json::to_vec(&value)?;
assert_eq!(encoded, br#"{"symbol":"AAPL","quantity":100}"#);
assert_eq!(json::from_slice(&encoded)?, value);
assert_eq!(json::from_str(r#"{"symbol":"AAPL","quantity":100}"#)?, value);
assert_eq!(value.get_key_str("symbol"), Some(&Value::from("AAPL")));

## Values JSON has no syntax for

In [ ]:
use yggdryl::{Value, json};

let value = Value::from_mapping([
    (Value::from("payload"), Value::from(vec![0_u8, 1, 255])),
    (Value::from("big"), Value::U128(1_u128 << 127)),
    (Value::from("ratio"), Value::from(f64::NAN)),
])?;

let encoded = json::to_vec(&value)?;
assert!(String::from_utf8(encoded.clone())?.contains(r#""$yggdryl""#));

let decoded = json::from_slice(&encoded)?;
assert_eq!(
    decoded.get_key_str("payload").and_then(Value::as_bytes),
    Some([0_u8, 1, 255].as_slice())
);
assert_eq!(
    decoded.get_key_str("big").and_then(Value::as_u128),
    Some(1_u128 << 127)
);
assert!(
    decoded
        .get_key_str("ratio")
        .and_then(Value::as_f64)
        .is_some_and(f64::is_nan)
);

## Readers and writers

In [ ]:
use std::io::Cursor;
use yggdryl::{Value, json};

let value = Value::from_mapping([(Value::from("symbol"), Value::from("AAPL"))])?;

let mut target = Vec::new();
json::to_writer(&mut target, &value)?;
assert_eq!(json::from_reader(Cursor::new(&target))?, value);

let error = json::from_str(r#"{"symbol":"AAPL"} 42"#).unwrap_err();
assert_eq!(
    error.to_string(),
    "invalid json data at byte 18: trailing characters after JSON value"
);

## Newline-delimited JSON

In [ ]:
use yggdryl::{Value, json};

let rows = [
    Value::from_mapping([(Value::from("id"), Value::from(1_u64))])?,
    Value::from_mapping([(Value::from("id"), Value::from(2_u64))])?,
];

let encoded = json::to_vec_all(&rows)?;
assert_eq!(encoded, b"{\"id\":1}\n{\"id\":2}\n");
assert_eq!(json::from_lines_slice(&encoded)?, rows);

// Blank and CRLF-terminated lines are skipped; two values on one are not.
assert_eq!(json::from_lines_str("{\"id\":1}\r\n\n{\"id\":2}\n")?, rows);
assert_eq!(
    json::from_lines_str("{\"id\":1} {\"id\":2}\n")
        .unwrap_err()
        .to_string(),
    "invalid json data at byte 9: trailing characters after JSON value"
);

// The writer never needs the rows materialized.
let mut target = Vec::new();
json::to_writer_all(&mut target, (0_u64..3).map(Value::from))?;
assert_eq!(target, b"0\n1\n2\n");

In [ ]:
use std::io::Cursor;
use yggdryl::{Value, json};

let mut source = Cursor::new(b"{\"id\":1}\n{\"id\":2}\n{\"id\":3}\n");
let mut rows = json::LinesReader::new(&mut source);

let first = rows.next().transpose()?.expect("one row");
assert_eq!(first.get_key_str("id"), Some(&Value::from(1_u64)));
assert_eq!(rows.byte_offset(), 9);
assert_eq!(rows.collect::<yggdryl::Result<Vec<_>>>()?.len(), 2);

// `Reader` splits on any JSON whitespace instead of on newlines.
let free = json::Reader::new(Cursor::new(b"1 2 3")).collect::<yggdryl::Result<Vec<_>>>()?;
assert_eq!(free, [Value::from(1_u64), Value::from(2_u64), Value::from(3_u64)]);

## Laying out a dump

In [ ]:
use yggdryl::generic::Value;
use yggdryl::text::Formatting;

let value = Value::from_mapping([
    (Value::String("id".into()), Value::I64(1)),
    (Value::String("tags".into()), Value::from_sequence([Value::String("a".into())])),
])?;

assert_eq!(yggdryl::json::to_vec(&value)?, br#"{"id":1,"tags":["a"]}"#);
assert_eq!(
    yggdryl::json::to_vec_with_formatting(&value, Formatting::indented(2))?,
    b"{\n  \"id\": 1,\n  \"tags\": [\n    \"a\"\n  ]\n}",
);

// Formatting changes bytes, never meaning.
assert_eq!(
    yggdryl::json::from_slice(
        &yggdryl::json::to_vec_with_formatting(&value, Formatting::indented(4))?,
    )?,
    value,
);

## Limits

In [ ]:
use yggdryl::{Limits, json};

let defaults = Limits::default();
assert_eq!(defaults.max_depth(), 128);
assert_eq!(defaults.max_input_bytes(), 64 * 1024 * 1024);
assert_eq!(defaults.max_nodes(), 1_000_000);
assert_eq!(defaults.max_documents(), 1_024);

let tight = Limits::new(2, 1024, 8, 1);
assert!(json::from_slice_with_limits(b"[[0]]", tight).is_ok());
assert_eq!(
    json::from_slice_with_limits(b"[[[0]]]", tight)
        .unwrap_err()
        .to_string(),
    "invalid json data at byte 2: nesting depth limit exceeded"
);

// No caller limit raises the parser's own ceiling.
let over = format!(
    "{}0{}",
    "[".repeat(json::MAX_PARSER_DEPTH + 1),
    "]".repeat(json::MAX_PARSER_DEPTH + 1)
);
let generous = Limits::new(4096, over.len(), 4096, 1);
assert!(
    json::from_str_with_limits(&over, generous)
        .unwrap_err()
        .to_string()
        .contains("parser hard limit of 384")
);

## Failures carry a byte offset

In [ ]:
use yggdryl::json;

// A duplicate is reported at the second key, not at the object.
assert_eq!(
    json::from_str(r#"{"symbol":"AAPL","symbol":"MSFT"}"#)
        .unwrap_err()
        .to_string(),
    "invalid json data at byte 17: JSON object contains a duplicate key"
);

// Row offsets are cumulative over the whole input, not per line.
assert_eq!(
    json::from_lines_str("{\"id\":1}\n{bad}\n")
        .unwrap_err()
        .to_string(),
    "invalid json data at byte 10: JSON object key must be a string"
);

## A compound filename carries the coding

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::text::{dump, load};
use yggdryl::{Url, Value};

let mut handle =
    Buffer::new().with_media_type(Url::from_str("file:///trades.json.gz")?.media_type());

let value = Value::from_mapping([(Value::from("symbol"), Value::from("AAPL"))])?;
dump(&mut handle, &value)?;

// The stored bytes really are gzip, and reading them back is symmetric.
assert_eq!(&handle.as_slice()[..2], &[0x1F, 0x8B]);
assert_eq!(load(&handle)?, value);